In [1]:
import scanpy as sc
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import scipy


from pydeseq2.dds import DeseqDataSet
from pydeseq2.default_inference import DefaultInference
from pydeseq2.ds import DeseqStats

In [4]:
data_annotated_path = "/home/bmb/haxx/working/ceisel_mumm/data/"
data = sc.read_h5ad(data_annotated_path + "full_annotations_leiden.h5ad")

data.shape

In [5]:
data.obs

In [6]:
data.layers['raw'] = np.array(data.layers['raw'].astype(int).todense())

# Pseudobulking

In [53]:
import anndata as ad
pseudobulks = []

n_pseduo_bulks = 1

for cell_type in sorted(list(set(data.obs['cell_type']))):
    if cell_type == 'trash':
        continue
    for condition in set(data.obs['exp_condition']):
        for time in sorted(list(set(data.obs['time']))):
            # print(cell_type,condition,time)
            mask = (
                (data.obs['cell_type'] == cell_type) & \
                (data.obs['exp_condition'] == condition) & \
                (data.obs['time'] == time) \
            )
            mask_split = np.random.randint(n_pseduo_bulks,size=mask.shape)
            submasks = [mask & (mask_split == i) for i in range(n_pseduo_bulks)]
            for submask in submasks:
                counts = data.layers['raw'][submask].sum(axis=0)
                annd = ad.AnnData(counts.reshape(1,-1))
                annd.obs['cell_type'] = cell_type
                annd.obs['exp_condition'] = condition
                annd.obs['time'] = time

                annd.obs['cell_type'] = annd.obs['cell_type'].astype('category')
                annd.obs['exp_condition'] = annd.obs['exp_condition'].astype('category')
                annd.obs['time'] = annd.obs['time'].astype('category')
                

                pseudobulks.append(annd)
            


In [54]:
pseudobulks = ad.concat(pseudobulks)
pseudobulks.obs_names_make_unique()

In [55]:
pseudobulks.obs

In [56]:
inference = DefaultInference(n_cpus=8)
dds = DeseqDataSet(
    counts=pseudobulks.X+1,
    metadata=pseudobulks.obs,
    # design="~exp_condition*cell_type",
    design="~ exp_condition + cell_type + time + exp_condition:time + exp_condition:cell_type",
    refit_cooks=True,
    inference=inference,
)

In [57]:
dds.obs["cell_type"].cat.categories

In [58]:
# Sets RGCs as the reference population for all relative changes

excluded_categories = list(dds.obs["cell_type"].cat.categories.unique())
excluded_categories.remove("RGCs")
dds.obs["cell_type"] = dds.obs["cell_type"].cat.reorder_categories([
    "RGCs",
    *excluded_categories,
]) 

In [59]:
dds.deseq2()

In [60]:
ds = DeseqStats(dds, contrast=["exp_condition", "Mtz", "Cntr"], inference=inference)

In [61]:
ds.summary()

### Volcano Plot

In [62]:
ds.LFC


In [63]:
plt.hist(-np.log10(ds.padj),bins=50)

In [64]:
plt.figure()
plt.scatter(
    ds.LFC['exp_condition[T.Mtz]'],
    # -np.log10(ds.padj),
    -np.log10(ds.p_values),
    # ds.statistics,
    s=1,
)
plt.show()

# Subset RGCs

In [65]:
pseudobulk_subset = pseudobulks[pseudobulks.obs['cell_type'] == "RGCs"]
pseudobulk_subset.var_names = data.var_names
inference = DefaultInference(n_cpus=8)
dds_sub = DeseqDataSet(
    counts=pseudobulk_subset.X+1,
    metadata=pseudobulk_subset.obs,
    # design="~ exp_condition + time + exp_condition:time",
    design="~ exp_condition + time",
    refit_cooks=True,
    inference=inference,
)


In [66]:
dds_sub.deseq2()

In [67]:
ds_sub = DeseqStats(dds_sub, contrast=["exp_condition", "Mtz", "Cntr"], inference=inference)

In [68]:
ds_sub.summary()

In [69]:
ds_sub.LFC

In [78]:
lfc_cutoff = 1
pval_cutoff = 1

padj_cutoff = 0.05

genes_to_mark = (
    (np.abs(ds_sub.LFC['exp_condition[T.Mtz]']) > lfc_cutoff) &
    (-np.log10(ds_sub.p_values) > pval_cutoff)
    # (-np.log10(ds_sub.padj) > padj_cutoff)
)
# np.abs(ds_sub.LFC['exp_condition[T.Mtz]'])


In [80]:
plt.figure(figsize=(10,10))
plt.title("DESeq2, RGCs only, Change in Injury compared to Control")

x = ds_sub.LFC['exp_condition[T.Mtz]']
y = -np.log10(ds_sub.p_values)
# y = -np.log10(ds_sub.padj)
plt.scatter(
    x,y,
    s=1,
)
ax = plt.gca()
offset = 0.05
for gene,x,y in zip(pseudobulk_subset.var_names[genes_to_mark],x[genes_to_mark],y[genes_to_mark]):
    ax.text(
        x + offset,y + offset,
        gene,fontsize=10)

ylims = ax.get_ylim()
xlims = ax.get_xlim()
plt.plot([lfc_cutoff,lfc_cutoff],ylims,'k--')
plt.plot([-lfc_cutoff,-lfc_cutoff],ylims,'k--')
plt.plot(xlims,[pval_cutoff,pval_cutoff],'k--')

plt.xlabel("Log2 Fold Change")
plt.ylabel("-Log10(p-value)")
plt.tight_layout()
plt.show()

# Signature Sets

In [72]:
# Selecting signature genes only 



signature = 'parthanatos'
genes_to_mark = data.var[signature]
genes_to_mark = np.array(genes_to_mark)

plt.figure(figsize=(10,10))
plt.title("DESeq2, RGCs only, Change in Injury compared to Control")

x = ds_sub.LFC['exp_condition[T.Mtz]']
y = -np.log10(ds_sub.p_values)
# y = -np.log10(ds_sub.padj)
plt.scatter(
    x,y,
    s=1,
)
ax = plt.gca()
offset = 0.05
for gene,x,y in zip(pseudobulk_subset.var_names[genes_to_mark],x[genes_to_mark],y[genes_to_mark]):
    ax.text(
        x + offset,y + offset,
        gene,fontsize=10)

ylims = ax.get_ylim()
xlims = ax.get_xlim()
plt.plot([lfc_cutoff,lfc_cutoff],ylims,'k--')
plt.plot([-lfc_cutoff,-lfc_cutoff],ylims,'k--')
plt.plot(xlims,[pval_cutoff,pval_cutoff],'k--')

plt.xlabel("Log2 Fold Change")
plt.ylabel("-Log10(p-value)")
plt.tight_layout()
plt.show()

In [85]:
for signature in ['parthanatos','necroptosis','apoptosis']:
    mask = np.array(data.var[signature])

    print("-"*100)
    print(f"Signature: {signature}")
    print(f"Total: \t{np.sum(ds_sub.LFC['exp_condition[T.Mtz]'][mask])}")
    print(f"\t\t{'\t'.join([f'{g[:6]},' for g in data.var_names[mask]])}")
    print(f"LFCs: \t\t{'\t'.join(np.around(ds_sub.LFC['exp_condition[T.Mtz]'][mask],3).astype(str))}")
    print(f"pvals: \t\t{'\t'.join(np.around(ds_sub.p_values[mask],3).astype(str))}")
    print(f"padjs: \t\t{'\t'.join(np.around(ds_sub.padj[mask],3).astype(str))}")

In [86]:
sc.pl.dotplot(data,parthanatos_signature,groupby="time")